# Boostrap cell (run to detect environment local/colab)

# PROJECT BOOTSTRAP

In [9]:
# ============================================================
# PROJECT BOOTSTRAP
#
# LOCAL:
#   - Dùng source code local
#   - Dùng data local
#
# COLAB:
#   - Mount Google Drive
#   - Clone/pull private GitHub repo vào /content
#   - Data persistent nằm trên Google Drive
#   - Working/cache nằm trên /content
#
# Yêu cầu với private GitHub repo:
#   - Nếu chạy trong Colab UI: có thể dùng Colab Secret GITHUB_TOKEN
#   - Nếu chạy Colab runtime từ VS Code:
#       bootstrap sẽ hỏi PAT bằng getpass() một lần/runtime
#
# Sau bootstrap có thể dùng:
#
#   PROJECT_ROOT
#   SRC_DIR
#   RAW_DIR
#   INTERIM_DIR
#   PROCESSED_DIR
#
#   WORK_ROOT
#   WORK_RAW_DIR
#   WORK_INTERIM_DIR
#   WORK_PROCESSED_DIR
#   DUCKDB_TEMP_DIR
#
#   update_code()
#   stage_file(...)
#   persist_file(...)
# ============================================================

from pathlib import Path

import base64
import os
import shutil
import subprocess
import sys

from getpass import getpass


# ============================================================
# 0. CONFIG
# ============================================================

REPO_OWNER = "lbngyn"
REPO_NAME = "Santander-Product-Recommendation"

REPO_URL = (
    f"https://github.com/"
    f"{REPO_OWNER}/"
    f"{REPO_NAME}.git"
)

# Persistent data trên Google Drive khi chạy Colab
DRIVE_DATA_ROOT = Path(
    "/content/drive/MyDrive/projects/santander/data"
)

# Dùng để tìm project root khi chạy local
PROJECT_MARKERS = [
    "src",
    "notebooks",
]


# ============================================================
# 1. DETECT ENVIRONMENT
# ============================================================

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

ENV = "colab" if IS_COLAB else "local"


# ============================================================
# 2. FIND LOCAL PROJECT ROOT
# ============================================================

def find_project_root(start: Path) -> Path:
    """
    Đi ngược từ current working directory
    cho tới khi tìm thấy project root.
    """

    current = start.resolve()

    while True:

        if any(
            (current / marker).exists()
            for marker in PROJECT_MARKERS
        ):
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        f"Không tìm thấy project root từ: {start}"
    )


# ============================================================
# 3. GITHUB AUTHENTICATION
# ============================================================

_GITHUB_TOKEN = None


def get_github_token():
    """
    Lấy GitHub PAT theo thứ tự:

    1. Token đã cache trong RAM
    2. Environment variable GITHUB_TOKEN
    3. Colab Secrets (nếu chạy trực tiếp trong Colab UI)
    4. getpass() fallback cho VS Code + Colab runtime

    Token không được ghi vào notebook hoặc Git remote URL.
    """

    global _GITHUB_TOKEN

    # --------------------------------------------------------
    # 1. Runtime cache
    # --------------------------------------------------------

    if _GITHUB_TOKEN:
        return _GITHUB_TOKEN


    # --------------------------------------------------------
    # 2. Environment variable
    # --------------------------------------------------------

    token = os.environ.get(
        "GITHUB_TOKEN"
    )

    if token:

        _GITHUB_TOKEN = token

        return token


    # --------------------------------------------------------
    # 3. Try Colab Secrets
    # --------------------------------------------------------

    if IS_COLAB:

        try:

            from google.colab import userdata

            token = userdata.get(
                "GITHUB_TOKEN"
            )

            if token:

                _GITHUB_TOKEN = token

                return token

            print("Colab Secret GITHUB_TOKEN is missing, empty, or not granted to this notebook.")
            print("Open the Secrets panel, verify the exact name, and enable Notebook access.")

        except Exception as error:
            # Expected khi dùng Colab runtime qua VS Code
            print(f"Colab Secret GITHUB_TOKEN is unavailable ({type(error).__name__}: {error}).")
            print("Check that this notebook is open in the Colab browser and Notebook access is enabled.")


    # --------------------------------------------------------
    # 4. VS Code + Colab fallback
    # --------------------------------------------------------

    print()
    print(
        "GitHub authentication required "
        "for private repository."
    )

    print(
        "Paste your GitHub Personal Access Token below."
    )

    print(
        "Token is kept only in the current runtime memory."
    )

    token = getpass(
        "GitHub PAT: "
    )

    if not token:

        raise RuntimeError(
            "GitHub token is required "
            "to access the private repository."
        )

    _GITHUB_TOKEN = token

    return token


def get_github_auth_header():
    """
    Tạo temporary Authorization header cho GitHub HTTPS.
    """

    token = get_github_token()

    credentials = (
        f"x-access-token:{token}"
    )

    encoded = base64.b64encode(
        credentials.encode("utf-8")
    ).decode("utf-8")

    return (
        f"AUTHORIZATION: basic {encoded}"
    )


# ============================================================
# 4. RUN GIT COMMAND
# ============================================================

def run_git(
    args,
    authenticated=False,
):
    """
    Chạy Git command.

    authenticated=True:
        dùng PAT qua temporary HTTP header.

    Token không được lưu vào origin URL.
    """

    command = [
        "git"
    ]

    if authenticated:

        auth_header = (
            get_github_auth_header()
        )

        command += [
            "-c",
            (
                "http.https://github.com/"
                f".extraheader={auth_header}"
            ),
        ]

    command += list(args)

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if result.stdout.strip():

        print(
            result.stdout.strip()
        )

    if result.returncode != 0:

        print()
        print("Git error:")

        print(
            result.stderr.strip()
        )

        raise RuntimeError(
            "Git command failed "
            f"with exit code {result.returncode}"
        )

    return result


# ============================================================
# 5. ENVIRONMENT SETUP
# ============================================================

if IS_COLAB:

    # --------------------------------------------------------
    # 5.1 MOUNT GOOGLE DRIVE
    # --------------------------------------------------------

    from google.colab import drive

    DRIVE_MOUNT = Path(
        "/content/drive"
    )

    if not (
        DRIVE_MOUNT / "MyDrive"
    ).exists():

        print(
            "Mounting Google Drive..."
        )

        drive.mount(
            str(DRIVE_MOUNT)
        )

    else:

        print(
            "Google Drive already mounted."
        )


    # --------------------------------------------------------
    # 5.2 PROJECT ROOT
    # --------------------------------------------------------

    PROJECT_ROOT = (
        Path("/content")
        / REPO_NAME
    )


    # --------------------------------------------------------
    # 5.3 CLEAN BROKEN CLONE
    # --------------------------------------------------------

    if (
        PROJECT_ROOT.exists()
        and not (
            PROJECT_ROOT / ".git"
        ).exists()
    ):

        print(
            "Found incomplete project directory."
        )

        print(
            "Removing broken clone..."
        )

        shutil.rmtree(
            PROJECT_ROOT
        )


    # --------------------------------------------------------
    # 5.4 CLONE / PULL
    # --------------------------------------------------------

    if not PROJECT_ROOT.exists():

        print(
            "Cloning private GitHub project..."
        )

        run_git(
            [
                "clone",
                REPO_URL,
                str(PROJECT_ROOT),
            ],
            authenticated=True,
        )

        print(
            "Repository cloned successfully."
        )

    else:

        print(
            "Project already exists."
        )

        print(
            "Pulling latest code..."
        )

        run_git(
            [
                "-C",
                str(PROJECT_ROOT),
                "pull",
                "--ff-only",
            ],
            authenticated=True,
        )


    # Persistent storage
    DATA_ROOT = DRIVE_DATA_ROOT

    # Fast temporary Colab disk
    WORK_ROOT = Path(
        "/content/cache"
    )


else:

    # ========================================================
    # LOCAL ENVIRONMENT
    # ========================================================

    PROJECT_ROOT = (
        find_project_root(
            Path.cwd()
        )
    )

    DATA_ROOT = (
        PROJECT_ROOT
        / "data"
    )

    WORK_ROOT = (
        PROJECT_ROOT
        / ".cache"
    )


# ============================================================
# 6. COMMON PATHS
# ============================================================

SRC_DIR = (
    PROJECT_ROOT
    / "src"
)

RAW_DIR = (
    DATA_ROOT
    / "raw"
)

INTERIM_DIR = (
    DATA_ROOT
    / "interim"
)

PROCESSED_DIR = (
    DATA_ROOT
    / "processed"
)


WORK_RAW_DIR = (
    WORK_ROOT
    / "raw"
)

WORK_INTERIM_DIR = (
    WORK_ROOT
    / "interim"
)

WORK_PROCESSED_DIR = (
    WORK_ROOT
    / "processed"
)

DUCKDB_TEMP_DIR = (
    WORK_ROOT
    / "duckdb_temp"
)


# ============================================================
# 7. CREATE DIRECTORIES
# ============================================================

directories = [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,

    WORK_RAW_DIR,
    WORK_INTERIM_DIR,
    WORK_PROCESSED_DIR,

    DUCKDB_TEMP_DIR,
]

for directory in directories:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 8. MAKE PROJECT IMPORTABLE
# ============================================================

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# ============================================================
# 9. UPDATE SOURCE CODE
# ============================================================

def update_code():
    """
    LOCAL:
        dùng trực tiếp source code local.

    COLAB:
        git pull latest commit từ private GitHub repo.
    """

    if not IS_COLAB:

        print(
            "Local environment: "
            "using local source code directly."
        )

        return

    print(
        "Pulling latest code from GitHub..."
    )

    run_git(
        [
            "-C",
            str(PROJECT_ROOT),
            "pull",
            "--ff-only",
        ],
        authenticated=True,
    )

    print(
        "Source code updated."
    )


# ============================================================
# 10. STAGE FILE
# ============================================================

def stage_file(
    source: Path,
    destination_dir: Path = None,
    overwrite: bool = False,
) -> Path:
    """
    Copy persistent file -> working storage.

    Colab:
        Drive -> /content/cache

    Local:
        data -> .cache
    """

    source = Path(
        source
    )

    if destination_dir is None:

        destination_dir = (
            WORK_INTERIM_DIR
        )

    destination_dir = Path(
        destination_dir
    )

    if not source.exists():

        raise FileNotFoundError(
            f"Source file không tồn tại: {source}"
        )

    destination_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination = (
        destination_dir
        / source.name
    )

    if (
        destination.exists()
        and not overwrite
    ):

        print(
            f"Using staged file: {destination}"
        )

        return destination

    print(
        f"Staging:\n"
        f"  {source}\n"
        f"  -> {destination}"
    )

    shutil.copy2(
        source,
        destination,
    )

    return destination


# ============================================================
# 11. PERSIST FILE
# ============================================================

def persist_file(
    source: Path,
    destination_dir: Path = None,
    overwrite: bool = True,
) -> Path:
    """
    Copy working output -> persistent storage.

    Colab:
        /content/cache -> Drive

    Local:
        .cache -> data/processed
    """

    source = Path(
        source
    )

    if destination_dir is None:

        destination_dir = (
            PROCESSED_DIR
        )

    destination_dir = Path(
        destination_dir
    )

    if not source.exists():

        raise FileNotFoundError(
            f"Source file không tồn tại: {source}"
        )

    destination_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination = (
        destination_dir
        / source.name
    )

    if destination.exists():

        if not overwrite:

            print(
                f"File already exists: {destination}"
            )

            return destination

        destination.unlink()

    print(
        f"Persisting:\n"
        f"  {source}\n"
        f"  -> {destination}"
    )

    shutil.copy2(
        source,
        destination,
    )

    return destination


# ============================================================
# 12. AUTO-RELOAD
# ============================================================

try:

    ipython = get_ipython()

    ipython.run_line_magic(
        "load_ext",
        "IPython.extensions.autoreload",
    )

    ipython.run_line_magic(
        "autoreload",
        "2",
    )

    print(
        "Autoreload: enabled"
    )

except Exception as e:

    print(
        f"Autoreload: disabled ({e})"
    )


# ============================================================
# 13. VALIDATION
# ============================================================

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"PROJECT_ROOT không tồn tại: "
        f"{PROJECT_ROOT}"
    )


if not SRC_DIR.exists():

    print()
    print(
        f"WARNING: src/ not found: {SRC_DIR}"
    )


# ============================================================
# 14. SUMMARY
# ============================================================

print()
print(
    "=" * 70
)

print(
    "PROJECT ENVIRONMENT"
)

print(
    "=" * 70
)

print(
    f"Environment      : {ENV}"
)

print(
    f"Project root     : {PROJECT_ROOT}"
)

print(
    f"Source           : {SRC_DIR}"
)

print()

print(
    "Persistent data"
)

print(
    f"  Raw            : {RAW_DIR}"
)

print(
    f"  Interim        : {INTERIM_DIR}"
)

print(
    f"  Processed      : {PROCESSED_DIR}"
)

print()

print(
    "Working storage"
)

print(
    f"  Work root      : {WORK_ROOT}"
)

print(
    f"  DuckDB temp    : {DUCKDB_TEMP_DIR}"
)

print(
    "=" * 70
)

# ------------------------------------------------------------
# 15. RUNTIME-ONLY CONFIGURATION
# ------------------------------------------------------------
# All later cells use the same environment variables on local
# and Colab. Only this bootstrap knows the active platform.
os.chdir(PROJECT_ROOT)

if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

    from google.colab import userdata

    def load_colab_secret(name: str, required: bool = True) -> None:
        try:
            value = userdata.get(name)
        except Exception as error:
            raise RuntimeError(f'Cannot read Colab Secret {name}. Enable notebook access for it.') from error
        if not value and required:
            raise RuntimeError(f'Missing required Colab Secret: {name}')
        if value:
            os.environ[name] = value

    for secret_name in ('GCP_SERVICE_ACCOUNT_JSON', 'GOOGLE_CLOUD_PROJECT', 'GCS_BUCKET'):
        load_colab_secret(secret_name)
    load_colab_secret('GCS_RAW_PREFIX', required=False)
    load_colab_secret('GCS_CHECKPOINT_PREFIX', required=False)
else:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / '.env')

print(f'Notebook ready: {ENV} | {PROJECT_ROOT}')

Google Drive already mounted.
Cloning private GitHub project...

GitHub authentication required for private repository.
Paste your GitHub Personal Access Token below.
Token is kept only in the current runtime memory.
Repository cloned successfully.
Autoreload: disabled (No module named 'imp')


PROJECT ENVIRONMENT
Environment      : colab
Project root     : /content/Santander-Product-Recommendation
Source           : /content/Santander-Product-Recommendation/src

Persistent data
  Raw            : /content/drive/MyDrive/projects/santander/data/raw
  Interim        : /content/drive/MyDrive/projects/santander/data/interim
  Processed      : /content/drive/MyDrive/projects/santander/data/processed

Working storage
  Work root      : /content/cache
  DuckDB temp    : /content/cache/duckdb_temp


# Start Ingest

In [ ]:
from src.data.ingest import config_from_environment, run_full_ingest

INGEST_CONFIG = config_from_environment(chunksize=250_000)

In [ ]:
ingest_result = run_full_ingest(
    config=INGEST_CONFIG,
    csv_filename='train_ver2.csv',
    parquet_filename='train.parquet',
)

ingest_result